In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)
    !git pull -q

    !pip install -q mlflow

In [2]:
import mlflow

!mlflow db upgrade sqlite:///mlflow.db
mlflow.set_tracking_uri('sqlite:///mlflow.db')

2026/07/11 06:02:01 INFO mlflow.store.db.utils: Updating database tables


In [ ]:
ARCHS = sorted(e.name[:-len('_Training')] for e in mlflow.MlflowClient().search_experiments()
              if e.name.endswith('_Training'))

client = mlflow.MlflowClient()

# diagnostikam achvena rom kaggle-is qcevas yvelaze ukot test-formis holdout-i
# (wmae_holdout) imeorebs da ara CV. amitom arqiteqturebi romlebsac es metrika
# aqvt am metrikit dardeba ertmanets; dzvel modelebs (holdout-is gareshe)
# mxolod mashin vadarebt CV-it (fold 1-2), tu holdout arc ert models ar aqvs
def arch_metrics(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{arch}_CV'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    if not runs:
        return None
    m = runs[0].data.metrics
    folds = client.get_metric_history(runs[0].info.run_id, 'wmae')
    mature = [x.value for x in folds if x.step >= 1]
    cv = sum(mature) / len(mature) if mature else m.get('wmae_mean')
    return {'holdout': m.get('wmae_holdout'), 'cv': cv}

scores = {arch: metrics for arch in ARCHS if (metrics := arch_metrics(arch)) is not None}
with_holdout = {a: s['holdout'] for a, s in scores.items() if s['holdout'] is not None}
if with_holdout:
    best_arch = min(with_holdout, key=with_holdout.get)
else:
    best_arch = min(scores, key=lambda a: scores[a]['cv'])
scores, best_arch

In [ ]:
MODEL_NAME = 'walmart-sales-model'

def final_run(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id,
                              filter_string=f"attributes.run_name = '{arch}_Final'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    return runs[0] if runs else None

def register(run, name):
    existing = [m.name for m in client.search_registered_models()]
    cur = client.get_registered_model(name).latest_versions[0] if name in existing else None
    if cur is None or cur.run_id != run.info.run_id:
        mlflow.register_model(f'runs:/{run.info.run_id}/model', name)

# yovel arqiteqturis sauketeso (Final) pipeline registrdeba tavisi saxelit,
# rom Model Registry-shi yvela arqiteqtura chans da chamoitvirtos saxelit
for arch in ARCHS:
    run = final_run(arch)
    if run is None:
        continue
    try:
        register(run, f'walmart-{arch}')
    except Exception as e:
        print('skip', arch, ':', e)

# saerto sauketeso modeli (holdout-it) icvleba kanonikuri saxelit walmart-sales-model
register(final_run(best_arch), MODEL_NAME)

[m.name for m in client.search_registered_models()]

In [5]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Register best model"
    !git push

[main 4050261] Register best model
 1 file changed, 0 insertions(+), 0 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 585 bytes | 585.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   f72a10a..4050261  main -> main
